# DAViD Sketch Notebook

This notebook orchestrates the training pipeline. All logic lives in:
- `config.py`: paths, hyperparameters
- `dataset.py`: `OptimizedVideoDataset`, data loaders
- `model.py`: `ClassificationHead`, `SwiGLU`, transforms
- `encoder.py`: ViFi-CLIP feature extractor loader
- `train.py`: training loop, validation, seed setting
- `clip/`: vendored CLIP architecture (for weight compatibility)

## 1. Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!git clone https://github.com/aitf-its-tim3-dfk/david.git

In [ ]:
%cd david
%pip install ftfy regex decord scikit-learn

## 2. Load Feature Extractor

In [ ]:
from config import CLIP_ARCH, CLASS_NAMES, VIFICLIP_CHECKPOINT
from encoder import load_feature_extractor

feature_extractor = load_feature_extractor(
    arch=CLIP_ARCH,
    class_names=CLASS_NAMES,
    checkpoint_path=VIFICLIP_CHECKPOINT,
).cuda()

## 3. Prepare Data

In [ ]:
import os
from config import METADATA_CSV, BASE_DIR, BATCH_SIZE, VAL_SPLIT, NUM_WORKERS
from config import TRAIN_SIZE, VAL_SIZE
from config import INPUT_DIM, NUM_CLASSES, LEARNING_RATE, NUM_EPOCHS, BEST_MODEL_SAVE_PATH, HEAD_TYPE

In [ ]:
from model import build_transform
from dataset import get_train_val_loaders

preprocess = build_transform(224)
train_loader, val_loader, val_files = get_train_val_loaders(
    transform=preprocess,
    csv_path=METADATA_CSV,
    base_dir=BASE_DIR,
    val_split=VAL_SPLIT,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

## 4. Train

In [ ]:
from config import USE_AMP, LR_SCHEDULER, PATIENCE
from train import set_seed, run_training

set_seed(42)

classifier = run_training(
    feature_extractor=feature_extractor,
    train_loader=train_loader,
    val_loader=val_loader,
    input_dim=INPUT_DIM,
    num_classes=NUM_CLASSES,
    head_type=HEAD_TYPE,
    lr=LEARNING_RATE,
    num_epochs=NUM_EPOCHS,
    save_path=BEST_MODEL_SAVE_PATH,
    use_amp=USE_AMP,
    lr_scheduler=LR_SCHEDULER,
    patience=PATIENCE,
)

## 5. Evaluate

from evaluate import evaluate
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
metrics = evaluate(classifier, feature_extractor, val_loader, val_files, device)

In [ ]:
## 6. Save & Cleanup

In [ ]:
from google.colab import drive, runtime

drive.flush_and_unmount()
runtime.unassign()